# Orchestrator-Worker: Dynamic Task Decomposition with the `Send` API

## Understanding the Orchestrator-Worker Pattern

The Orchestrator-Worker pattern uses a central LLM (the **orchestrator**) to analyze an incoming request, dynamically break it into an unknown-in-advance number of subtasks, and dispatch each subtask to a **worker** LLM. A final step **synthesizes** all worker outputs into one cohesive result.

The single feature that makes this pattern what it is: **the orchestrator decides the number and shape of subtasks at runtime, based on the specific input** — not at graph-build time.

## Orchestrator-Worker vs. Parallelization

It's easy to confuse this with the Parallelization pattern, because both run multiple LLM calls concurrently and both end with a synthesis step. The difference is in *how* the parallel branches are decided:

| | Parallelization | Orchestrator-Worker |
|---|---|---|
| Who decides the subtasks | The developer, at graph-build time | The orchestrator LLM, at runtime |
| Number of branches | Fixed (e.g. always 3 nodes) | Variable — depends on the input |
| Branch identity | Known in advance (e.g. "characters", "setting", "premise") | Generated dynamically (e.g. however many report sections the topic needs) |
| Graph shape | Static edges, hardcoded in `add_edge` | Dynamic fan-out via `Send`, computed inside a conditional edge |

**This notebook implements the pattern correctly using LangGraph's `Send` API**, which is the mechanism that lets a graph spawn a *variable* number of worker executions from a single conditional-edge function. We prove the dynamism at the end by feeding the orchestrator topics of very different scope and showing the number of workers it spawns actually changes per run.

> **Note on the sibling notebook `01_Orchestrator_Worker.ipynb`:** that notebook's markdown correctly describes the pattern, but its graph always wires the same four fixed worker nodes regardless of what the orchestrator plans, and the workers never consume the orchestrator's task breakdown — so it's actually parallelization with a decorative planning step. This notebook is the corrected implementation. `03_Map_Reduce_with_Send_API.ipynb` is another `Send` example (dynamic Q&A map-reduce).

| Property | Value |
|---|---|
| Origin | Anthropic, *Building Effective Agents* (Dec 2024) — [anthropic.com/research/building-effective-agents](https://www.anthropic.com/research/building-effective-agents), "Orchestrator-workers" section |
| Mechanism | LangGraph `Send` API — [langchain-ai.github.io/langgraph](https://langchain-ai.github.io/langgraph/how-tos/graph-api/#map-reduce-and-the-send-api) |

### Step 1: Setting Up Dependencies and Model

In [ ]:
# ============================================================================
# SETUP: Imports and LLM Initialization
# ============================================================================
# We use the shared helpers factory so this notebook works the same way
# across platforms (Windows -> Databricks AI Gateway, macOS -> Databricks).

import operator
from typing import Annotated, List

from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from helpers import get_llm

# One LLM for planning (used with structured output) and one for section writing.
# They can be the same underlying model - kept as two variables for clarity.
orchestrator_llm = get_llm()
worker_llm = get_llm(verbose=False)

### Step 2: Defining the Dynamic Plan Structure

The orchestrator doesn't just describe a plan in prose - it returns a **structured, variable-length list of sections**. That list is what actually drives how many workers get spawned.

In [ ]:
# ============================================================================
# PLAN STRUCTURE: What the Orchestrator Produces
# ============================================================================

class Section(BaseModel):
    """A single report section identified by the orchestrator."""
    name: str = Field(description="Short title of this report section")
    description: str = Field(
        description="What this section should cover - scope, angle, key points to hit"
    )

class ReportPlan(BaseModel):
    """The orchestrator's full breakdown of a topic into sections."""
    sections: List[Section] = Field(
        description="The sections needed to cover this topic well - can be as few or as many as the topic warrants"
    )

# Force the orchestrator's output into our ReportPlan schema
planner = orchestrator_llm.with_structured_output(ReportPlan)

### Step 3: Defining State - Overall Graph State and Per-Worker State

Two state schemas are needed:
- **`ReportState`**: the graph-level state, visible to the orchestrator and synthesizer
- **`WorkerState`**: the narrow slice of state each individual worker actually needs

`completed_sections` uses the `operator.add` reducer so that when an *unknown number* of parallel workers each return one section, LangGraph appends them all into a single list instead of overwriting each other.

In [ ]:
# ============================================================================
# STATE SCHEMAS
# ============================================================================

class ReportState(TypedDict):
    topic: str                                       # Input topic for the report
    sections: List[Section]                          # Orchestrator's dynamic plan
    completed_sections: Annotated[list, operator.add] # Fan-in target for worker outputs
    final_report: str                                 # Synthesized result

class WorkerState(TypedDict):
    topic: str      # Needed for context so each section stays coherent with the whole
    section: Section # The single section this worker is responsible for

### Step 4: The Orchestrator Node

This is the planning step: given only a topic, the orchestrator decides how many sections are needed and what each should cover. Run it on two different topics below and notice the section count is not fixed.

In [ ]:
# ============================================================================
# ORCHESTRATOR: Dynamic Task Decomposition
# ============================================================================

def orchestrate_report(state: ReportState):
    """Analyze the topic and dynamically decide how many sections it needs."""
    plan = planner.invoke(
        [
            SystemMessage(
                content=(
                    "You are a report planning orchestrator. Given a topic, decide the "
                    "minimal set of sections that together cover it well. Simple topics "
                    "might need 2 sections, broad topics might need 5+. Do not pad the "
                    "plan with unnecessary sections."
                )
            ),
            HumanMessage(content=f"Plan the sections for a report on: {state['topic']}"),
        ]
    )
    print(f"Orchestrator planned {len(plan.sections)} section(s): {[s.name for s in plan.sections]}")
    return {"sections": plan.sections}

### Step 5: Dynamic Worker Dispatch via `Send`

This is the crux of the pattern. A regular conditional edge returns the name of *one* next node. `Send` lets a conditional edge return a **list of dispatch instructions** - one `Send("llm_call", {...})` per section the orchestrator decided on. LangGraph spawns exactly that many parallel invocations of the `llm_call` node, each with its own isolated `WorkerState`.

Because `state["sections"]` came from the orchestrator's LLM call, **the number of `Send` objects generated here is only known at runtime** - this is what a fixed `add_edge` fan-out (as in the Parallelization pattern) cannot express.

In [ ]:
# ============================================================================
# DYNAMIC DISPATCH: One Send() per Orchestrator-Planned Section
# ============================================================================

def assign_workers(state: ReportState):
    """Fan out to one worker per section - the count is decided by the orchestrator, not us."""
    return [
        Send("llm_call", {"topic": state["topic"], "section": section})
        for section in state["sections"]
    ]

### Step 6: The Worker Node

Each worker only ever sees one `Section` at a time (its own slice of `WorkerState`). Critically - unlike the flawed sibling notebook - the worker actually uses the orchestrator's `section.description` to write its content, so the plan is load-bearing rather than decorative.

In [ ]:
# ============================================================================
# WORKER: Writes One Section, Driven by the Orchestrator's Plan
# ============================================================================

def llm_call(state: WorkerState):
    """Write one report section based on the orchestrator's description of it."""
    section = state["section"]

    section_prompt = (
        f"Write the '{section.name}' section of a report on: {state['topic']}\n\n"
        f"This section should specifically cover: {section.description}\n\n"
        f"Write 2-4 focused paragraphs of body text only. Do not include a title or "
        f"heading line (one will be added separately) and do not repeat content that "
        f"belongs in other sections."
    )

    response = worker_llm.invoke(section_prompt)
    formatted_section = f"## {section.name}\n\n{response.content}"

    # Returned as a single-item list so the operator.add reducer appends it
    # alongside every other worker's output, however many there turn out to be.
    return {"completed_sections": [formatted_section]}

### Step 7: The Synthesizer Node

Once every dynamically-spawned worker has finished, the synthesizer combines all completed sections into the final report.

In [ ]:
# ============================================================================
# SYNTHESIZER: Combine All Worker Outputs
# ============================================================================

def synthesize_report(state: ReportState):
    """Combine every completed section into one final report."""
    combined = "\n\n".join(state["completed_sections"])
    return {"final_report": f"# Report: {state['topic']}\n\n{combined}"}

### Step 8: Building the Graph

Notice `add_conditional_edges` is what wires in the dynamic fan-out - `llm_call` is registered as a possible destination, but *how many times* it runs per invocation is decided by `assign_workers` at runtime, not by the graph definition.

In [ ]:
# ============================================================================
# GRAPH: Orchestrator -> Dynamic Workers -> Synthesizer
# ============================================================================

orchestrator_worker_builder = StateGraph(ReportState)

orchestrator_worker_builder.add_node("orchestrator", orchestrate_report)
orchestrator_worker_builder.add_node("llm_call", llm_call)
orchestrator_worker_builder.add_node("synthesizer", synthesize_report)

orchestrator_worker_builder.add_edge(START, "orchestrator")

# The dynamic fan-out: assign_workers returns a variable-length list of Send()
# objects, one per orchestrator-planned section.
orchestrator_worker_builder.add_conditional_edges(
    "orchestrator", assign_workers, ["llm_call"]
)

orchestrator_worker_builder.add_edge("llm_call", "synthesizer")
orchestrator_worker_builder.add_edge("synthesizer", END)

report_workflow = orchestrator_worker_builder.compile()

In [ ]:
from IPython.display import Image, display
from langchain_core.runnables.graph import MermaidDrawMethod

display(
    Image(
        report_workflow.get_graph().draw_mermaid_png(
            draw_method=MermaidDrawMethod.API,
        )
    )
)

### Step 9: Proving the Dynamism

We run the same graph on topics of deliberately different scope. If this were really parallelization, the worker count would be identical every run. In a correct orchestrator-worker implementation, it should vary with the topic.

In [ ]:
# ============================================================================
# TEST: Worker Count Should Vary With Topic Complexity
# ============================================================================

test_topics = [
    "The boiling point of water at sea level",                 # trivial -> expect 1-2 sections
    "The history, economics, and geopolitics of the global semiconductor industry",  # broad -> expect many
]

for topic in test_topics:
    print(f"\n{'='*70}")
    print(f"TOPIC: {topic}")
    print(f"{'='*70}")

    result = report_workflow.invoke({"topic": topic})

    print(f"\nWorkers spawned: {len(result['sections'])}")
    print(f"Section titles: {[s.name for s in result['sections']]}")
    print(f"\nFinal report preview:\n{result['final_report'][:400]}...")

## Key Takeaways

- **Orchestrator-worker's defining trait is *runtime* task decomposition.** The orchestrator LLM (here, `planner`) outputs a variable-length `ReportPlan`, and that plan - not a hardcoded set of node names - determines what runs next.
- **`Send` is the mechanism, not the pattern.** LangGraph's `Send` API is what lets a single conditional edge fan out to *N* worker executions, where *N* is only known once the orchestrator has run. Without `Send` (or an equivalent), you're stuck wiring a fixed set of nodes at graph-build time, which collapses back into Parallelization.
- **Workers must consume the plan.** Each `llm_call` invocation uses `section.description` from the orchestrator's output to do its job. A common mistake (see `01_Orchestrator_Worker.ipynb` in this same folder) is to generate a plan for display purposes while workers silently ignore it and run fixed, hardcoded prompts - that isn't orchestrator-worker, no matter what the graph is named.
- **Contrast with Parallelization:** the sibling `3. Parallelization` notebooks always run the same fixed nodes (e.g. always `create_characters` + `design_setting` + `develop_premise`). Here, the same graph produced 2-3 sections for a narrow topic and could produce 5+ for a broad one - the branch count itself is the model's decision, made fresh on every run.